# 02. 칼질 + 양자화 + 평가 (L4, 약 3시간 / 약 5~6유닛)
| 단계 | 만드는 GGUF | 확인하는 것 |
|---|---|---|
| 기준 | `original-Q6_K`, `original-Q4_K_M` | 출발점 (Q6_K는 거의 무손실, Q4_K_M은 배포 형식 비교용) |
| 칼질 | `ko-k48-Q6_K`, `ko-k32-Q6_K` | 전문가를 줄이면 한국어가 얼마나 떨어지나 |
| 양자화 | `ko-k32-Q4_K_M`, `-Q3_K_M`, `-Q2_K` | 비트를 줄이면 얼마나 떨어지나 |

변형마다 GGUF 만들기 → 객관식·퍼플렉시티 평가 → 속도·전력 측정 → 파일 삭제 순서로 진행합니다.
결과는 끝날 때마다 드라이브에 저장되므로, 세션이 끊기면 다시 실행하세요. 끝난 것은 건너뜁니다.
**디스크가 약 100GB 필요합니다.** 아래 `df` 결과를 확인하세요.

In [ ]:
# 저장소 받기 + 패키지 설치
BRANCH = "ccr-a463bea6-bpbyrh"   # main에 합친 뒤에는 "main"
REPO = "github.com/b1bibig/Cfinetune"
import os
try:  # 저장소가 비공개면 코랩 왼쪽 열쇠(보안 비밀)에 GH_TOKEN 을 넣어 두세요
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_tok}@{REPO}"
except Exception:
    REPO_URL = f"https://{REPO}"
if not os.path.exists("/content/Cfinetune"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/Cfinetune
%cd /content/Cfinetune
!git pull -q
%pip install -q -r requirements-colab.txt

In [ ]:
# 구글 드라이브: 결과(results/)를 세션이 끊겨도 남도록 저장
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, sys
DRIVE = "/content/drive/MyDrive/cfinetune"
os.makedirs(DRIVE, exist_ok=True)

def pull_results():
    if os.path.exists(f"{DRIVE}/results"):
        shutil.copytree(f"{DRIVE}/results", "results", dirs_exist_ok=True)

def push_results():
    shutil.copytree("results", f"{DRIVE}/results", dirs_exist_ok=True)
    print("드라이브에 결과 저장함")

pull_results()
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU 메모리 {mem:.0f}GB")
assert mem >= 20, "런타임 > 런타임 유형 변경에서 L4 를 고르세요"
!df -h /content | tail -1

In [ ]:
# llama.cpp 빌드 (처음 약 15분, 이후에는 드라이브 캐시에서 복사)
!bash scripts/build_llama_cpp.sh /content/llama.cpp {DRIVE}/cache

In [ ]:
from _common import resolve_model_dir, load_config
cfg = load_config()
SRC = resolve_model_dir(cfg["model"]["id"])   # 원본 BF16 (약 44GB)
GGUF, PRUNED = "/content/gguf", "/content/models/pruned"
REF, DEPLOY = cfg["reference_quant"], cfg["deploy_quant"]

def done(name):
    import glob
    return os.path.exists(f"results/eval/{name}.json") and bool(glob.glob(f"results/bench/{name}@*.json"))

def run_variant(hf_dir, stem, quants):
    todo = [q for q in quants if not done(f"{stem}-{q}")]
    if not todo:
        print("건너뜀:", stem); return
    QS = " ".join(todo)
    !python scripts/03_make_gguf.py --hf-dir {hf_dir} --name {stem} --out-dir {GGUF} --quant {QS}
    for q in todo:
        g = f"{GGUF}/{stem}-{q}.gguf"
        !python scripts/05_eval.py --gguf {g} --name {stem}-{q}
        !python scripts/06_bench.py --gguf {g} --name {stem}-{q}
        if stem == "original" and q == cfg["pc"]["copy_original"]:   # 내 PC 측정용 (docs/pc.md)
            os.makedirs(f"{DRIVE}/gguf", exist_ok=True)
            shutil.copy(g, f"{DRIVE}/gguf/")
        os.remove(g)
        push_results()

## 기준: 원본

In [ ]:
run_variant(SRC, "original", [REF, DEPLOY])

## 칼질 (+ 기준 변형은 양자화 비교까지)

In [ ]:
base = cfg["quantization"]["base_variant"]
for v in cfg["pruning"]["variants"]:
    name = v["name"]
    quants = [REF] + (cfg["quantization"]["types"] if name == base else [])
    if all(done(f"{name}-{q}") for q in quants):
        print("건너뜀:", name); continue
    !python scripts/02_prune.py --variant {name} --src {SRC} --out-dir {PRUNED}
    run_variant(f"{PRUNED}/{name}", name, quants)
    shutil.rmtree(f"{PRUNED}/{name}")

## 중간 결과 표

In [ ]:
!python scripts/07_econ.py
push_results()